# 05. 모델링

`04_eda`까지 생성한 파생변수 포함 데이터셋을 사용해 실거래가 예측 모델을 학습한다.

진행 순서:

- 최종 파생변수 데이터셋 로드 및 모델 학습용 컬럼 정리
- 시간 분할과 랜덤 분할 기준으로 기준 모델 성능 비교
- 가격대별 오차, log-price 실험, 변수 중요도, 예측값 비교를 통해 이후 모델 선정 근거 마련

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(PROJECT_ROOT / '.cache'))
os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count() or 1))
(PROJECT_ROOT / '.matplotlib').mkdir(exist_ok=True)
(PROJECT_ROOT / '.cache').mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
%matplotlib inline

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

DATA_PATH = PROJECT_ROOT / 'data/processed/seoul_apt_trade_2025_features.csv'
MODELING_DATA_PATH = PROJECT_ROOT / 'data/processed/modeling_dataset.csv'
SCORE_PATH = PROJECT_ROOT / 'reports/model_scores.csv'
PRICE_BAND_MAE_PATH = PROJECT_ROOT / 'reports/model_price_band_mae.csv'
PREDICTION_PATH = PROJECT_ROOT / 'reports/model_test_predictions.csv'
FIGURE_DIR = PROJECT_ROOT / 'reports/figures'
IMPORTANCE_PATH = FIGURE_DIR / 'model_feature_importance.png'

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 100)

## 1. 최종 파생변수 데이터 로드

모델링은 `03_feature_engineering`에서 만든 최종 데이터셋을 기준으로 한다.

In [ ]:
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
print(f'rows: {len(df):,}')
print(f'columns: {len(df.columns):,}')
df.head()

## 2. 모델 학습용 데이터셋 구성

좌표 생성, 주소 확인, 사후 검증용 컬럼은 모델 입력에서 제외한다. 타깃은 실거래가 총액인 `price_10k_krw`를 사용한다.

`price_per_m2_10k_krw`는 타깃인 거래가를 면적으로 나눈 값이라 누수 가능성이 있으므로 입력 변수에서 제외한다.

In [ ]:
target_col = 'price_10k_krw'

numeric_features = [
    'area_m2',
    'floor',
    'built_year',
    'age',
    'contract_year',
    'contract_month',
    'contract_day',
    'distance_to_cbd_km',
    'distance_to_ybd_km',
    'distance_to_gbd_km',
    # 'nearest_business_district_distance_km',
    'nearest_subway_distance_km',
    # 'hospital_count_within_1km',
    'nearest_hospital_distance_km',
    'large_mart_count_within_1km',
]

categorical_features = [
    'gu',
    'law_dong',
    # 'apartment_name',
    # 'nearest_business_district',
]

required_columns = [target_col, 'contract_date', *numeric_features, *categorical_features]
modeling_df = df[required_columns].copy()
modeling_df['contract_date'] = pd.to_datetime(modeling_df['contract_date'])

missing_before = modeling_df.isna().sum().sort_values(ascending=False)
missing_before[missing_before > 0]

In [ ]:
rows_before = len(modeling_df)
modeling_df = modeling_df.dropna(subset=[target_col, *numeric_features, *categorical_features]).copy()
rows_after = len(modeling_df)

modeling_df.to_csv(MODELING_DATA_PATH, index=False, encoding='utf-8-sig')

print(f'모델링 데이터 저장: {MODELING_DATA_PATH}')
print(f'제거된 행: {rows_before - rows_after:,}')
print(f'최종 행 수: {rows_after:,}')
modeling_df.head()

## 3. 학습/테스트 데이터 분할

실제 예측 상황과 비슷하게 거래일 기준 시간 분할을 사용한다. 2025년 1월부터 10월까지는 학습, 2025년 11월부터 12월까지는 테스트로 둔다.

In [ ]:
split_date = pd.Timestamp('2025-11-01')
train_df = modeling_df[modeling_df['contract_date'] < split_date].copy()
test_df = modeling_df[modeling_df['contract_date'] >= split_date].copy()

X_train = train_df[numeric_features + categorical_features]
y_train = train_df[target_col]
X_test = test_df[numeric_features + categorical_features]
y_test = test_df[target_col]

split_summary = pd.DataFrame({
    'dataset': ['train', 'test'],
    'start_date': [train_df['contract_date'].min(), test_df['contract_date'].min()],
    'end_date': [train_df['contract_date'].max(), test_df['contract_date'].max()],
    'rows': [len(train_df), len(test_df)],
    'target_mean': [y_train.mean(), y_test.mean()],
    'target_median': [y_train.median(), y_test.median()],
})
split_summary

## 4. 모델 파이프라인 정의

설치된 기본 라이브러리만 사용하기 위해 `scikit-learn` 모델로 기준선을 만든다.

- `LinearRegression`: 선형회귀
- `Ridge`: L2 규제가 적용된 선형회귀
- `Lasso`: L1 규제가 적용된 선형회귀
- `DecisionTreeRegressor`: 단일 결정 트리 회귀
- `RandomForestRegressor`: 랜덤 포레스트 회귀
- `ExtraTreesRegressor`: 엑스트라 트리 회귀
- `GradientBoostingRegressor`: 그래디언트 부스팅 회귀
- `HistGradientBoostingRegressor`: 히스토그램 기반 그래디언트 부스팅 회귀

In [ ]:
onehot_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False), categorical_features),
    ],
    remainder='drop',
)

ordinal_preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ]), categorical_features),
    ],
    remainder='drop',
)

models = {
    'linear_regression': Pipeline([
        ('preprocess', onehot_preprocessor),
        ('model', LinearRegression()),
    ]),
    'ridge': Pipeline([
        ('preprocess', onehot_preprocessor),
        ('model', Ridge(alpha=1.0)),
    ]),
    'lasso': Pipeline([
        ('preprocess', onehot_preprocessor),
        ('model', Lasso(alpha=1.0, max_iter=10000)),
    ]),
    'decision_tree': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', DecisionTreeRegressor(
            max_depth=18,
            min_samples_leaf=5,
            random_state=42,
        )),
    ]),
    'random_forest': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', RandomForestRegressor(
            n_estimators=120,
            max_depth=18,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    'extra_trees': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', ExtraTreesRegressor(
            n_estimators=160,
            max_depth=24,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    'gradient_boosting': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', GradientBoostingRegressor(
            n_estimators=160,
            learning_rate=0.06,
            max_depth=4,
            min_samples_leaf=3,
            random_state=42,
        )),
    ]),
    'hist_gradient_boosting': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', HistGradientBoostingRegressor(
            max_iter=180,
            learning_rate=0.06,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            random_state=42,
        )),
    ]),
}

linear_model_names = ['linear_regression', 'ridge', 'lasso']
overall_model_names = ['linear_regression', 'decision_tree', 'random_forest', 'extra_trees', 'gradient_boosting', 'hist_gradient_boosting']


## 5. 모델 학습 및 평가

평가 지표는 회귀 문제에서 자주 쓰는 `MAE`, `RMSE`, `R2`를 사용한다. 금액 단위는 원본 타깃과 같은 만 원이다.

In [ ]:
def evaluate_regression(y_true, y_pred):
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred),
    }


def fit_and_evaluate_models(X_train, y_train, X_test, y_test, split_strategy):
    scores = []
    fitted = {}

    for model_name, model in models.items():
        print(f'training: {split_strategy} / {model_name}')
        fitted_model = clone(model)
        fitted_model.fit(X_train, y_train)
        fitted[model_name] = fitted_model

        train_pred = fitted_model.predict(X_train)
        test_pred = fitted_model.predict(X_test)

        for dataset_name, y_true, y_pred in [
            ('train', y_train, train_pred),
            ('test', y_test, test_pred),
        ]:
            scores.append({
                'split_strategy': split_strategy,
                'model': model_name,
                'dataset': dataset_name,
                **evaluate_regression(y_true, y_pred),
            })

    return pd.DataFrame(scores), fitted


def make_test_prediction_df(fitted, X_test, y_test, split_strategy):
    rows = []
    for model_name, model in fitted.items():
        pred = model.predict(X_test)
        rows.append(pd.DataFrame({
            'split_strategy': split_strategy,
            'model': model_name,
            'actual_price_10k_krw': y_test.to_numpy(),
            'predicted_price_10k_krw': pred,
            'absolute_error_10k_krw': np.abs(y_test.to_numpy() - pred),
        }))
    return pd.concat(rows, ignore_index=True)

In [ ]:
score_df, fitted_models = fit_and_evaluate_models(
    X_train,
    y_train,
    X_test,
    y_test,
    split_strategy='time_2025_01_10_train_11_12_test',
)

test_prediction_df = make_test_prediction_df(
    fitted_models,
    X_test,
    y_test,
    split_strategy='time_2025_01_10_train_11_12_test',
)

In [ ]:
score_df.to_csv(SCORE_PATH, index=False, encoding='utf-8-sig')
print(f'성능표 저장: {SCORE_PATH}')
score_df.sort_values(['split_strategy', 'dataset', 'rmse'])

## 6. 랜덤 80:20 분할 성능 비교

전체 데이터를 섞은 뒤 80%는 훈련셋, 20%는 테스트셋으로 사용한다. 시간 분할보다 일반적인 교차 검증 상황에 가까운 비교용 결과다.

In [ ]:
X = modeling_df[numeric_features + categorical_features]
y = modeling_df[target_col]

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

random_split_summary = pd.DataFrame({
    'dataset': ['train_random', 'test_random'],
    'rows': [len(X_train_random), len(X_test_random)],
    'target_mean': [y_train_random.mean(), y_test_random.mean()],
    'target_median': [y_train_random.median(), y_test_random.median()],
})
random_split_summary

In [ ]:
random_score_df, random_fitted_models = fit_and_evaluate_models(
    X_train_random,
    y_train_random,
    X_test_random,
    y_test_random,
    split_strategy='random_80_20',
)

random_prediction_df = make_test_prediction_df(
    random_fitted_models,
    X_test_random,
    y_test_random,
    split_strategy='random_80_20',
)
test_prediction_df = pd.concat([test_prediction_df, random_prediction_df], ignore_index=True)

score_df = pd.concat([score_df, random_score_df], ignore_index=True)
score_df.to_csv(SCORE_PATH, index=False, encoding='utf-8-sig')
print(f'성능표 업데이트: {SCORE_PATH}')
score_df.sort_values(['split_strategy', 'dataset', 'rmse'])

## 7. 분할 방식별 테스트 성능 비교

시간 분할과 랜덤 분할 결과를 RMSE 기준으로 나란히 비교한다.

In [ ]:
test_scores = score_df[score_df['dataset'].eq('test')].sort_values(['split_strategy', 'rmse']).reset_index(drop=True)
test_scores

### 7-1. Linear model family comparison

`LinearRegression`, `Ridge`, and `Lasso` belong to the same linear model family.
Their performance differences are very small, so the y-axis is zoomed in with exact values annotated.
Only `linear_regression` represents the linear family in the overall model comparison.

In [ ]:
linear_test_scores = (
    test_scores[test_scores['model'].isin(linear_model_names)]
    .sort_values(['split_strategy', 'model'])
    .reset_index(drop=True)
)
display(linear_test_scores[['split_strategy', 'model', 'mae', 'rmse', 'r2']])

_split_order = [
    s for s in ['time_2025_01_10_train_11_12_test', 'random_80_20']
    if s in test_scores['split_strategy'].unique()
]
_split_labels = {
    'time_2025_01_10_train_11_12_test': 'time split',
    'random_80_20': 'random split',
}
_model_colors = {
    'linear_regression': '#4C78A8',
    'ridge':             '#59A14F',
    'lasso':             '#F58518',
}

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
fig.subplots_adjust(hspace=0.5, wspace=0.35)

for col, split in enumerate(_split_order):
    split_data = test_scores[test_scores['split_strategy'] == split].set_index('model')
    label = _split_labels[split]

    for row, metric in enumerate(['mae', 'rmse']):
        ax = axes[row, col]
        vals = [split_data.loc[m, metric] for m in linear_model_names if m in split_data.index]
        names = [m for m in linear_model_names if m in split_data.index]
        colors = [_model_colors[m] for m in names]
        x = np.arange(len(names))

        ax.bar(x, vals, color=colors, alpha=0.85, width=0.45)

        y_min = min(vals)
        y_max = max(vals)
        pad = max((y_max - y_min) * 4, y_max * 0.001)
        ax.set_ylim(y_min - pad, y_max + pad * 3)

        for i, v in enumerate(vals):
            ax.text(i, v + pad * 0.2, f'{v:,.0f}', ha='center', va='bottom',
                    fontsize=9, fontweight='bold')

        ax.set_xticks(x)
        ax.set_xticklabels(names, fontsize=9)
        ax.set_title(f'{metric.upper()} — {label}\n(y-axis zoomed)', fontsize=10)
        ax.set_ylabel(f'{metric.upper()} (10k KRW)')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, p: f'{v:,.0f}'))
        ax.grid(axis='y', alpha=0.25)

fig.suptitle('Linear Model Family: LinearRegression vs Ridge vs Lasso', fontsize=12, fontweight='bold')
fig.tight_layout()
display(fig)
fig.savefig(PROJECT_ROOT / 'reports/figures/05_linear_models_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig)


In [ ]:
# Best model summary
time_best = test_scores[test_scores['split_strategy'].str.startswith('time')].iloc[0]
random_best = test_scores[test_scores['split_strategy'].str.startswith('random')].iloc[0]

print('=== Best Model (Time Split) ===')
print(f"  Model : {time_best['model']}")
print(f"  MAE   : {time_best['mae']:,.0f} 만원")
print(f"  RMSE  : {time_best['rmse']:,.0f} 만원")
print(f"  R²    : {time_best['r2']:.4f}")
print()
print('=== Best Model (Random Split, Reference) ===')
print(f"  Model : {random_best['model']}")
print(f"  MAE   : {random_best['mae']:,.0f} 만원")
print(f"  RMSE  : {random_best['rmse']:,.0f} 만원")
print(f"  R²    : {random_best['r2']:.4f}")

In [ ]:
# train vs test 성능 비교 (과적합 확인)
overfit_check = (
    score_df[score_df['model'].isin(overall_model_names)]
    .pivot_table(
        index=['split_strategy', 'model'],
        columns='dataset',
        values=['mae', 'rmse', 'r2'],
    )
    .round(0)
)

split_order = [
    split for split in ['time_2025_01_10_train_11_12_test', 'random_80_20']
    if split in score_df['split_strategy'].unique()
]

fig, axes = plt.subplots(
    len(split_order),
    2,
    figsize=(14, 5 * len(split_order)),
    squeeze=False,
)

for row_idx, split_strategy in enumerate(split_order):
    split_scores = score_df[
            score_df['split_strategy'].eq(split_strategy)
            & score_df['model'].isin(overall_model_names)
        ]
    for col_idx, metric in enumerate(['mae', 'rmse']):
        ax = axes[row_idx, col_idx]
        metric_scores = split_scores.pivot(index='model', columns='dataset', values=metric)
        model_order = [model_name for model_name in overall_model_names if model_name in metric_scores.index]
        metric_scores = metric_scores.reindex(model_order)

        x = np.arange(len(metric_scores.index))
        width = 0.35
        ax.bar(x - width / 2, metric_scores['train'], width, label='train', color='#4C78A8', alpha=0.8)
        ax.bar(x + width / 2, metric_scores['test'], width, label='test', color='#F58518', alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(metric_scores.index, rotation=15, ha='right')
        ax.set_title(f'{metric.upper()} - Train vs Test ({split_strategy})')
        ax.set_ylabel(f'{metric.upper()} (10k KRW)')
        ax.legend()
        ax.grid(axis='y', alpha=0.25)

fig.tight_layout()
display(fig)
fig.savefig(PROJECT_ROOT / 'reports/figures/05_train_vs_test.png', dpi=150, bbox_inches='tight')
plt.close(fig)
overfit_check




In [ ]:
plot_df = (
    test_scores[test_scores['model'].isin(overall_model_names)]
    .sort_values(['split_strategy', 'rmse'], ascending=[True, False])
    .copy()
)
plot_df['label'] = plot_df['split_strategy'] + ' / ' + plot_df['model']
fig, ax = plt.subplots(figsize=(9, max(5, len(plot_df) * 0.35)))
ax.barh(plot_df['label'], plot_df['rmse'], color='#4C78A8')
ax.set_xlabel('RMSE (10k KRW)')
ax.set_ylabel('split / model')
ax.set_title('Test RMSE by Split Strategy')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
display(fig)
plt.close(fig)

## 8. 가격대별 MAE 확인

전체 MAE만 보면 저가·고가 구간 중 어느 구간에서 오차가 큰지 알기 어렵다. 테스트셋의 실제 거래가를 가격 구간으로 나누고, 구간별 평균 절대 오차를 확인한다.

In [ ]:
price_bins = [0, 50_000, 100_000, 150_000, 200_000, 300_000, np.inf]
price_labels = [
    '<=5eok',
    '5-10eok',
    '10-15eok',
    '15-20eok',
    '20-30eok',
    '>30eok',
]

test_prediction_df['price_band'] = pd.cut(
    test_prediction_df['actual_price_10k_krw'],
    bins=price_bins,
    labels=price_labels,
    include_lowest=True,
)

price_band_mae = (
    test_prediction_df
    .groupby(['split_strategy', 'model', 'price_band'], observed=True)
    .agg(
        rows=('absolute_error_10k_krw', 'size'),
        actual_mean=('actual_price_10k_krw', 'mean'),
        pred_mean=('predicted_price_10k_krw', 'mean'),
        mae=('absolute_error_10k_krw', 'mean'),
    )
    .reset_index()
    .sort_values(['split_strategy', 'model', 'price_band'])
)

price_band_mae.to_csv(PRICE_BAND_MAE_PATH, index=False, encoding='utf-8-sig')
print(f'가격대별 MAE 저장: {PRICE_BAND_MAE_PATH}')
price_band_mae

In [ ]:
best_models = test_scores.groupby('split_strategy').first().reset_index()[['split_strategy', 'model']]
best_price_band_mae = price_band_mae.merge(best_models, on=['split_strategy', 'model'])
best_price_band_mae

In [ ]:
fig, axes = plt.subplots(1, len(best_models), figsize=(14, 4.5), sharey=True)
if len(best_models) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, best_models.iterrows()):
    plot_df = best_price_band_mae[
        best_price_band_mae['split_strategy'].eq(row['split_strategy'])
        & best_price_band_mae['model'].eq(row['model'])
    ]
    ax.bar(plot_df['price_band'].astype(str), plot_df['mae'], color='#4C78A8')
    ax.set_title(f"{row['split_strategy']} / {row['model']}")
    ax.set_xlabel('Actual price band')
    ax.tick_params(axis='x', rotation=35)
    ax.grid(axis='y', alpha=0.25)
axes[0].set_ylabel('MAE (10k KRW)')
fig.tight_layout()
display(fig)
plt.close(fig)

## 9. log-price 실험

트리/트리 앙상블 후보 모델을 기준으로 원가격을 직접 예측한 경우와 `log1p(price)`를 예측한 뒤 다시 가격 단위로 되돌린 경우를 비교한다. 이 실험은 최종 성능표에 저장하지 않고, log 변환이 실제로 도움이 되는지 확인하는 용도로만 사용한다.

In [ ]:
def fit_log_price_model(base_model, X_train, y_train, X_test):
    log_model = clone(base_model)
    log_model.fit(X_train, np.log1p(y_train))
    log_pred = np.expm1(log_model.predict(X_test))
    return log_model, np.maximum(log_pred, 0)


def make_log_experiment(model_name, split_strategy, fitted_raw_model, X_train, y_train, X_test, y_test):
    raw_pred = fitted_raw_model.predict(X_test)
    _, log_pred = fit_log_price_model(models[model_name], X_train, y_train, X_test)

    rows = []
    prediction_frames = []
    for target_type, pred in [('raw_price', raw_pred), ('log_price', log_pred)]:
        rows.append({
            'model': model_name,
            'split_strategy': split_strategy,
            'target_type': target_type,
            **evaluate_regression(y_test, pred),
        })
        prediction_frames.append(pd.DataFrame({
            'model': model_name,
            'split_strategy': split_strategy,
            'target_type': target_type,
            'actual_price_10k_krw': y_test.to_numpy(),
            'predicted_price_10k_krw': pred,
            'absolute_error_10k_krw': np.abs(y_test.to_numpy() - pred),
        }))

    return pd.DataFrame(rows), pd.concat(prediction_frames, ignore_index=True)

In [ ]:
log_experiment_parts = []
log_prediction_parts = []
log_experiment_model_names = ['decision_tree', 'random_forest', 'extra_trees', 'gradient_boosting', 'hist_gradient_boosting']

for model_name in log_experiment_model_names:
    log_time_scores, log_time_predictions = make_log_experiment(
        model_name,
        'time_2025_01_10_train_11_12_test',
        fitted_models[model_name],
        X_train,
        y_train,
        X_test,
        y_test,
    )
    log_random_scores, log_random_predictions = make_log_experiment(
        model_name,
        'random_80_20',
        random_fitted_models[model_name],
        X_train_random,
        y_train_random,
        X_test_random,
        y_test_random,
    )
    log_experiment_parts.extend([log_time_scores, log_random_scores])
    log_prediction_parts.extend([log_time_predictions, log_random_predictions])

log_experiment_scores = pd.concat(log_experiment_parts, ignore_index=True)
log_experiment_predictions = pd.concat(log_prediction_parts, ignore_index=True)

In [ ]:
log_experiment_scores.sort_values(['split_strategy', 'model', 'target_type'])

In [ ]:
log_experiment_predictions['price_band'] = pd.cut(
    log_experiment_predictions['actual_price_10k_krw'],
    bins=price_bins,
    labels=price_labels,
    include_lowest=True,
)

log_price_band_mae = (
    log_experiment_predictions
    .groupby(['split_strategy', 'model', 'target_type', 'price_band'], observed=True)
    .agg(
        rows=('absolute_error_10k_krw', 'size'),
        actual_mean=('actual_price_10k_krw', 'mean'),
        pred_mean=('predicted_price_10k_krw', 'mean'),
        mae=('absolute_error_10k_krw', 'mean'),
    )
    .reset_index()
    .sort_values(['split_strategy', 'model', 'target_type', 'price_band'])
)

log_price_band_mae

In [ ]:
fig, axes = plt.subplots(
    len(log_experiment_model_names),
    2,
    figsize=(14, max(8, len(log_experiment_model_names) * 3)),
    sharey=True,
)

split_strategies = log_experiment_scores['split_strategy'].unique()
for row_idx, model_name in enumerate(log_experiment_model_names):
    for col_idx, split_strategy in enumerate(split_strategies):
        ax = axes[row_idx, col_idx]
        plot_df = log_price_band_mae[
            log_price_band_mae['model'].eq(model_name)
            & log_price_band_mae['split_strategy'].eq(split_strategy)
        ]
        pivot_df = plot_df.pivot(index='price_band', columns='target_type', values='mae')
        x = np.arange(len(pivot_df.index))
        width = 0.38
        ax.bar(x - width / 2, pivot_df['raw_price'], width=width, label='raw_price', color='#4C78A8')
        ax.bar(x + width / 2, pivot_df['log_price'], width=width, label='log_price', color='#F58518')
        ax.set_title(f'{model_name} / {split_strategy}')
        ax.set_xticks(x)
        ax.set_xticklabels(pivot_df.index.astype(str), rotation=35, ha='right')
        ax.set_xlabel('Actual price band')
        ax.grid(axis='y', alpha=0.25)

for row_idx in range(len(log_experiment_model_names)):
    axes[row_idx, 0].set_ylabel('MAE (10k KRW)')
axes[0, 0].legend()
fig.tight_layout()
display(fig)
plt.close(fig)

## 10. 변수 중요도 확인

이 단계의 목적은 최종 모델 선택이 아니라 알고리즘 이해다. `DecisionTreeRegressor`, `RandomForestRegressor`, `ExtraTreesRegressor`, `GradientBoostingRegressor` 네 모델의 변수 중요도를 나란히 비교하면, 트리 깊이와 앙상블 방식에 따라 각 알고리즘이 어떤 변수를 다르게 강조하는지 확인할 수 있다. 최종 후보 선별은 이후 섹션 11에서 성능 기준으로 진행한다.

범주형 변수는 순서형 인코딩 기준으로 변수 단위 중요도를 해석한다. `HistGradientBoostingRegressor`는 동일한 native `feature_importances_` 속성을 제공하지 않으므로 07 단계의 Permutation Importance로 해석을 보완한다.


In [ ]:
importance_model_names = ['decision_tree', 'random_forest', 'extra_trees', 'gradient_boosting']
importance_frames = []

for model_name in importance_model_names:
    importance_model = fitted_models[model_name].named_steps['model']
    importance_frames.append(pd.DataFrame({
        'model': model_name,
        'feature': numeric_features + categorical_features,
        'importance': importance_model.feature_importances_,
    }))

importance_df = (
    pd.concat(importance_frames, ignore_index=True)
    .sort_values(['model', 'importance'], ascending=[True, False])
)

importance_df.to_csv(PROJECT_ROOT / 'reports/model_feature_importance.csv', index=False, encoding='utf-8-sig')
importance_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=False)
axes = axes.ravel()

for ax, model_name in zip(axes, importance_model_names):
    top_importance = (
        importance_df[importance_df['model'].eq(model_name)]
        .head(15)
        .sort_values('importance')
    )
    ax.barh(top_importance['feature'], top_importance['importance'], color='#59A14F')
    ax.set_xlabel('importance')
    ax.set_title(model_name)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='x', alpha=0.25)

for ax in axes[len(importance_model_names):]:
    ax.axis('off')

fig.suptitle('Native Feature Importance by Tree-Based Model', fontsize=12)
fig.tight_layout()
display(fig)
fig.savefig(IMPORTANCE_PATH, dpi=150, bbox_inches='tight')
print(f'변수 중요도 그림 저장: {IMPORTANCE_PATH}')
plt.close(fig)


## 11. 예측/잔차 상세 비교

전체 성능 비교에서는 배운 모델 전체를 평가하지만, 예측값·잔차·가격대별 오차 구조는 시간 분할 테스트 RMSE 기준 상위 2개 모델만 상세 비교한다. 시간 분할은 실제 예측 상황에 더 가까우며, 상위 2개로 제한하면 최종 후보와 대안 후보의 차이를 명확하게 해석할 수 있다.

In [ ]:
time_model_ranking = (
    test_scores[
        test_scores['split_strategy'].str.startswith('time')
        & test_scores['model'].isin(overall_model_names)
    ]
    .sort_values('rmse')
    .reset_index(drop=True)
)
prediction_model_names = time_model_ranking['model'].head(2).tolist()
print(f'상세 예측 비교 대상: {prediction_model_names}')
prediction_info_cols = [
    'contract_date',
    'gu',
    'law_dong',
    'apartment_name',
    'area_m2',
    'floor',
    'built_year',
    'age',
]


def make_prediction_comparison(fitted, X_data, y_data, source_df, split_strategy):
    available_info_cols = [col for col in prediction_info_cols if col in source_df.columns]
    prediction_df = source_df.loc[X_data.index, available_info_cols].copy()
    prediction_df.insert(0, 'split_strategy', split_strategy)
    prediction_df['actual_price_10k_krw'] = y_data.to_numpy()
    prediction_df['actual_price_eok'] = prediction_df['actual_price_10k_krw'] / 10_000

    for model_name in prediction_model_names:
        pred = fitted[model_name].predict(X_data)
        prediction_df[f'{model_name}_pred_price_10k_krw'] = pred
        prediction_df[f'{model_name}_pred_price_eok'] = pred / 10_000
        prediction_df[f'{model_name}_abs_error_10k_krw'] = np.abs(y_data.to_numpy() - pred)
        prediction_df[f'{model_name}_abs_error_eok'] = prediction_df[f'{model_name}_abs_error_10k_krw'] / 10_000

    return prediction_df


time_prediction_comparison = make_prediction_comparison(
    fitted_models,
    X_test,
    y_test,
    modeling_df,
    'time_2025_01_10_train_11_12_test',
)

random_prediction_comparison = make_prediction_comparison(
    random_fitted_models,
    X_test_random,
    y_test_random,
    modeling_df,
    'random_80_20',
)

prediction_comparison = pd.concat(
    [time_prediction_comparison, random_prediction_comparison],
    ignore_index=True,
)

prediction_comparison.to_csv(PREDICTION_PATH, index=False, encoding='utf-8-sig')
print(f'예측 결과 저장: {PREDICTION_PATH}')
prediction_comparison.head()

In [ ]:
prediction_summary = []
for split_strategy, group in prediction_comparison.groupby('split_strategy'):
    for model_name in prediction_model_names:
        prediction_summary.append({
            'split_strategy': split_strategy,
            'model': model_name,
            'mean_abs_error_10k_krw': group[f'{model_name}_abs_error_10k_krw'].mean(),
            'median_abs_error_10k_krw': group[f'{model_name}_abs_error_10k_krw'].median(),
            'mean_abs_error_eok': group[f'{model_name}_abs_error_eok'].mean(),
            'median_abs_error_eok': group[f'{model_name}_abs_error_eok'].median(),
        })

prediction_summary_df = pd.DataFrame(prediction_summary)
prediction_summary_df.sort_values(['split_strategy', 'mean_abs_error_10k_krw'])

In [ ]:
# 예측 샘플 확인용 셀이다.
# 제출용 실행에서 표 출력을 생략하려면 SHOW_PREDICTION_SAMPLE = False로 둔다.
SHOW_PREDICTION_SAMPLE = False

if SHOW_PREDICTION_SAMPLE:
    info_cols = [
        'split_strategy', 'contract_date', 'gu', 'law_dong',
        'apartment_name', 'area_m2', 'floor',
        'actual_price_10k_krw', 'actual_price_eok',
    ]
    pred_cols = [
        col for model_name in prediction_model_names
        for col in [f'{model_name}_pred_price_eok', f'{model_name}_abs_error_eok']
        if col in prediction_comparison.columns
    ]
    display(
        prediction_comparison[info_cols + pred_cols]
        .sample(10, random_state=42)
        .sort_values('split_strategy')
    )


### 가격대별 평균 잔차

가격 구간별로 예측값과 실제값의 평균 잔차(예측가 - 실제가)를 비교한다. 0 기준선 위쪽은 과대 예측, 아래쪽은 과소 예측이다.

In [ ]:
time_comparison = prediction_comparison[
    prediction_comparison['split_strategy'].str.startswith('time')
].copy()

time_comparison['price_band'] = pd.cut(
    time_comparison['actual_price_eok'],
    bins=[0, 5, 10, 15, 20, 30, float('inf')],
    labels=['<=5', '5-10', '10-15', '15-20', '20-30', '>30'],
    include_lowest=True,
)

bar_colors = ['#4C78A8', '#F58518']

price_bands = ['<=5', '5-10', '10-15', '15-20', '20-30', '>30']

residual_rows = []
for model_name in prediction_model_names:
    time_comparison[f'{model_name}_residual_eok'] = (
        time_comparison[f'{model_name}_pred_price_eok'] - time_comparison['actual_price_eok']
    )
    band_mean = time_comparison.groupby('price_band', observed=True)[f'{model_name}_residual_eok'].mean()
    for band, val in band_mean.items():
        residual_rows.append({'price_band': band, 'model': model_name, 'mean_residual': val})

residual_df = pd.DataFrame(residual_rows)
band_counts = time_comparison.groupby('price_band', observed=True).size().reindex(price_bands)
band_actual_mean = time_comparison.groupby('price_band', observed=True)['actual_price_eok'].mean().reindex(price_bands)

n_models = len(prediction_model_names)
x = range(len(price_bands))
width = 0.13

fig, ax = plt.subplots(figsize=(13, 5))

for idx, model_name in enumerate(prediction_model_names):
    vals = residual_df[residual_df['model'] == model_name].set_index('price_band').reindex(price_bands)['mean_residual']
    offset = (idx - (n_models - 1) / 2) * width
    ax.bar([v + offset for v in x], vals.values, width, label=model_name, color=bar_colors[idx % len(bar_colors)], alpha=0.85)

ax.axhline(0, color='red', linewidth=1, linestyle='--')
ax.set_xticks(list(x))
ax.set_xticklabels([
    f'{b}\nn={int(cnt)}, avg={avg:.1f}'
    for b, cnt, avg in zip(price_bands, band_counts.values, band_actual_mean.values)
])
ax.set_xlabel('Actual Price Band (100M KRW)')
ax.set_ylabel('Mean Residual (Predicted - Actual, 100M KRW)')
ax.set_title('Mean Residual by Price Band — Time Split Test Set')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.25)

fig.tight_layout()
display(fig)
fig.savefig(PROJECT_ROOT / 'reports/figures/05_residual_by_price_band.png', dpi=150, bbox_inches='tight')
plt.close(fig)


In [ ]:
def predict_apartment_price(feature_row, fitted, model_names=prediction_model_names, actual_target=None):
    if isinstance(feature_row, dict):
        input_df = pd.DataFrame([feature_row])
    else:
        input_df = feature_row.copy()

    input_df = input_df[numeric_features + categorical_features]
    result = pd.DataFrame(index=input_df.index)
    if actual_target is not None:
        if isinstance(actual_target, pd.Series):
            actual_values = actual_target.reindex(input_df.index).to_numpy()
        else:
            actual_values = np.asarray(actual_target)
        result['actual_price_10k_krw'] = actual_values
        result['actual_price_eok'] = result['actual_price_10k_krw'] / 10_000
    for model_name in model_names:
        pred = fitted[model_name].predict(input_df)
        result[f'{model_name}_pred_price_10k_krw'] = pred
        result[f'{model_name}_pred_price_eok'] = pred / 10_000
    return result

# 아래 예시는 시간 분할 테스트셋의 첫 번째 행을 사용한다.
# 직접 예측해보고 싶으면 example_features의 값을 수정한 뒤 이 셀을 다시 실행한다.
example_features = X_test.iloc[[0]].copy()
predict_apartment_price(example_features, fitted_models, actual_target=y_test)

## 12. 정리

05 단계에서는 수업에서 배운 지도학습 모델을 회귀 문제에 맞춰 비교한다. 비교와 해석 범위는 다음 기준으로 나눈다.

- **전체 성능 비교**: `linear_regression`, `decision_tree`, `random_forest`, `extra_trees`, `gradient_boosting`, `hist_gradient_boosting`을 비교한다. 선형회귀 계열에서는 `linear_regression`만 대표로 포함한다.
- **선형회귀 계열 비교**: `linear_regression`, `ridge`, `lasso`를 별도로 비교한다. 세 모델은 같은 선형회귀 계열이고 성능 차이가 작기 때문에 별도 그래프로 묶어 확인한다.
- **변수 중요도 확인**: native `feature_importances_`를 제공하는 `decision_tree`, `random_forest`, `extra_trees`, `gradient_boosting`만 비교한다. `hist_gradient_boosting`은 동일 속성을 제공하지 않으므로 07 단계의 Permutation Importance로 해석을 보완한다.
- **예측/잔차 상세 비교**: 시간 분할 테스트 RMSE 기준으로 `overall_model_names` 안에서 상위 2개 모델을 자동 선택한다. 어떤 모델이 포함될지는 실행 결과에 따라 결정되며, `hist_gradient_boosting`도 상위 2개 안에 드는 경우에만 포함된다.

05 단계의 목적은 최종 모델 확정이 아니라 후보 모델의 상대적 성능과 오차 구조를 확인하는 것이다. 06 단계에서는 05에서 확인한 트리/트리 앙상블 후보를 대상으로 변수 조합 비교와 하이퍼파라미터 탐색을 진행하고, 검증셋 RMSE가 가장 낮은 모델과 변수 조합을 최종 튜닝 대상으로 선택한다.
